# 2. Data Preparation

Standardize date fields, validate required business columns, and prepare the dataset for KPI and profitability analysis.

## Environment Setup

Define reusable project paths and display settings for a consistent workflow.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets" / "screenshots"
DATA_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## Load Prepared Data

Import the cleaned transaction file produced by the intake notebook.

In [ ]:
cleaned_path = DATA_DIR / "cleaned_data.csv"
df = pd.read_csv(cleaned_path, parse_dates=["Order Date", "Ship Date"])
df.head()

## Required Field Validation

Verify that all fields needed for KPI, margin, product, region, and customer analysis are present.

In [ ]:
df.columns = df.columns.str.strip()

required_columns = ["Order Date", "Ship Date", "Sales", "Profit", "Discount", "Category", "Sub-Category", "Region", "Segment"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available.")

## Date Features

Create year and month fields for time-based reporting.

In [ ]:
df["Order Year"] = df["Order Date"].dt.year

df["Order Month"] = df["Order Date"].dt.to_period("M").astype(str)

df[["Order Date", "Ship Date", "Order Year", "Order Month"]].head()

## Quality Controls

Run basic checks on sales, profit, missing values, and duplicate records before analysis.

In [ ]:
quality_checks = pd.DataFrame({
    "Check": ["Missing Sales", "Missing Profit", "Negative Sales", "Negative Profit", "Duplicate Rows"],
    "Result": [
        df["Sales"].isna().sum(),
        df["Profit"].isna().sum(),
        (df["Sales"] < 0).sum(),
        (df["Profit"] < 0).sum(),
        df.duplicated().sum(),
    ],
})
quality_checks

## Analysis Dataset Export

Save the analysis-ready dataset for performance analysis and reporting visuals.

In [ ]:
eda_path = DATA_DIR / "cleaned_data_for_EDA.csv"
df.to_csv(eda_path, index=False)
print(f"Saved analysis-ready dataset to: {eda_path}")